# US equities desk: research, read, execute

Start with **MSFT financial quality**, then change `CASE_FILE` to try another desk case.
The input files are all in the **cases** folder beside this notebook:

| File | Desk task | Answer |
| --- | --- | --- |
| [msft_fundamentals.json](cases/msft_fundamentals.json) | MSFT FY2023–FY2025 company review | Table + Markdown |
| [msft_risk_json.json](cases/msft_risk_json.json) | MSFT risk versus SPY | JSON only |
| [aapl_trend_strategy.json](cases/aapl_trend_strategy.json) | Fixed AAPL 200-session trend filter | Table + JSON + plain text |
| [sp500_momentum_screener.json](cases/sp500_momentum_screener.json) | Ranked current-universe US large-cap shortlist | Tables only |
| [qqq_treasury_regimes.json](cases/qqq_treasury_regimes.json) | QQQ/SPY returns and US Treasury yield changes | Table + Markdown |

Use **Epoch SDK (local prototype)**. Setup cells only prepare access; step 3 starts
real vendor/model and Backend work. These short desk requests are a new revision. Earlier live results used more
explicit requests and do not prove that this revision succeeds. There are no
pre-filled financial results or assumed winning strategies.

Each case has its own readable research and execution IDs. Matching inputs attach
to the same work; choose a new `ATTEMPT` when intentionally changing inputs or
refreshing the screener's membership snapshot. The screening price cutoff is fixed
at **2026-10-16**; it is not a historical S&P 500 membership backtest.

## 1. Connect to the running services

The prepared notebook server supplies your Epoch identity and both service URLs. Credentials are not stored in this notebook.

The functions imported from `notebook_support.py` are walkthrough helpers.
`connect()` calls the SDK's `EpochClient.connect(...)` with your Epoch login and
both service addresses, then keeps the resulting client in the variable `epoch`.
`check_services()` prints the installed SDK versions and reads the services' health
endpoints. `await` means this cell waits for those network checks to finish.
This setup step prepares access; the later `epoch.run(...)` call starts research.


In [ ]:
# Import the ordinary SDK types and the local walkthrough helpers.
from pathlib import Path
from IPython.display import JSON, display
from epoch_sdk.research import ResearchCompleted
from notebook_support import (
    connect, check_services, operation_name,
    wait_execution,
)
from case_support import (
    case_info, load_case, list_cases, read_case_tables, check_case_answer,
)

# Prepare the client using the Epoch login and service URLs supplied by the
# notebook server. This does not start a study or contact data vendors.
epoch = connect()
# Print package versions and check that Backend and EpochAI respond.
await check_services()

## 2. Choose the desk case and inspect its request

Each JSON file has a short **task**, a few **context** items describing data
and business assumptions, and **outputs** describing the requested answer.
Open a file from **cases** to inspect those three parts. Descriptions and units are part
of each output contract. This cell reads the local file; it starts no research.

Change `CASE_FILE` to one of the five base cases above. Notebook 03 handles the
output variants because they require an existing source execution. The default
`ATTEMPT = "desk-v2-01"` starts a separate operation for this shorter request
revision. Keep it unchanged to reconnect after a kernel restart.

In [ ]:
# Choose one of the five real desk scenarios. list_cases() shows all choices.
list_cases()
CASE_FILE = Path("cases/msft_fundamentals.json")
ATTEMPT = "desk-v2-01"  # Keep this to reconnect; change it for intentionally new work.

# Validate the local JSON against the SDK request models.
info = case_info(CASE_FILE)
assert info["kind"] == "research", "Use notebook 02 for clarification or 03 for output variants."
request = load_case(CASE_FILE)
# Use separate, case-specific names for research and saved-study execution.
# A different case never accidentally attaches to an earlier case's operation.
RESEARCH_ID = operation_name(info["id"] + "-" + ATTEMPT + "-research")
EXECUTION_ID = operation_name(info["id"] + "-" + ATTEMPT + "-execute")
print("Input case:", CASE_FILE.resolve())
print("Desk:", info["persona"])
print("Research ID:", RESEARCH_ID)
print("Execution ID:", EXECUTION_ID)
display(JSON(request.to_wire(), expanded=False))

## 3. Run the research session

This cell starts real research, or reconnects to the same readable research ID.
The SDK shows Epoch's live activity and the saved answer here. If Epoch needs
input, choose an option or write a reply and press **Send reply** inside this
cell. It continues the same checkpoint; no reply code or polling loop is needed.
Nothing is chosen automatically.

You can stop the cell or restart the kernel while the server continues. Reconnect
with this same ID. **Leave waiting** keeps an unanswered question on the server.
A data gap stays visible; there is no silent substitute or fabricated return.


In [ ]:
# One SDK call owns live progress, clarification widgets and resume.
# In Jupyter, run() selects the notebook view automatically.
# The result is still an ordinary typed Python result for subsequent analysis.
result = await epoch.run(request, operation_id=RESEARCH_ID)


## 4. Use the saved answer in Python

The session above has already displayed the requested table, JSON and text parts.
There is no second narrative to print. The returned result contains the exact
saved answer and source identities. Response tables read retained computed data
and do not force matching dashboard tables.

If you chose **Leave waiting**, rerun the research cell with the same ID to answer
and continue before running the following cells.


In [ ]:
# The session must have published its answer before these analysis cells continue.
assert isinstance(result, ResearchCompleted), "Continue the research session above first."
response = result.response  # Ordinary typed saved answer: declarations, parts, revision and source.


## 5. Check complete table reads and internal consistency

A transfer page size of two makes pagination visible for small tables. Every page
is read. JSON-only cases skip table calls. The checks match this case's formulas,
years, ranks and output declarations rather than assuming particular market returns.

These checks are **not an independent source audit**. Live review still needs to
reconcile vendor observations, inspect strategy timing/costs and assess the prose.

In [ ]:
# Read all rows of every requested table, with small pages for this walkthrough.
# Native int64 table cells remain decimal strings on the wire; checks decode
# them only where a count or rank is needed.
tables = await read_case_tables(epoch, result, page_size=2)
# Check output names/types, arithmetic and case-specific ordering.
check_case_answer(CASE_FILE, request, result, tables)

## 6. Execute the saved study directly through Backend

This is a separate action: it creates one new execution of the saved study under
`EXECUTION_ID`, without asking the supervisor to rediscover or rebuild it. Reuse
the case's exact period and native execution options, including the strategy costs.

This step returns execution status. It does not publish an updated answer or
refresh the original screener roster. Changed periods are only meaningful where
the saved study was authored to support them; the first walkthrough keeps them fixed.

In [ ]:
# Carry the exact case period and any native cost options into prepared execution.
assert result.reusable_study is not None
assert request.parameters is not None and request.parameters.period is not None
period = request.parameters.period
execution_options = request.parameters.execution
execution = await epoch.execute_saved(
    result.reusable_study,
    period=period,
    execution=execution_options,
    operation_id=EXECUTION_ID,
)
# This only reads the accepted Backend job until it completes.
execution = await wait_execution(epoch, EXECUTION_ID)
assert execution.execution.definition_id == result.reusable_study.study_id
print("Study:", execution.execution.definition_id)
print("New execution:", execution.execution.id)
print("Status:", execution.execution.status.value)

## 7. Repeat the call: attach to the same execution

Run this cell repeatedly. A new intended execution needs a new operation ID.

In [ ]:
# Repeat exactly the same study, period, costs and execution operation ID.
# The returned execution ID must match the preceding job.
same_execution = await epoch.execute_saved(
    result.reusable_study,
    period=period,
    execution=execution_options,
    operation_id=EXECUTION_ID,
)
assert same_execution.execution.id == execution.execution.id
print("Same execution:", same_execution.execution.id)

## 8. The research answer keeps its original source

Prepared execution returns an execution. The research answer remains pinned to its original run and revision. A new interpretation of new results is a separate answer-producing step.

In [ ]:
# Read the answer belonging to the original research operation.
# These checks confirm that its saved answer and revision have stayed the same.
# The prepared execution has a different run ID and does not replace that answer.
original = await epoch.research_status(operation_id=RESEARCH_ID)
assert isinstance(original, ResearchCompleted)
assert original.response.response_id == result.response.response_id
assert original.response.version == result.response.version
assert original.response.primary_execution_id != execution.execution.id
print("Original answer execution:", original.response.primary_execution_id)
print("Prepared execution:", execution.execution.id)
print("Pinned answer revision:", original.response.version)

## 9. Close this client

The services and saved results remain available. Notebook 02 demonstrates an
intentional confirmation and reconnect using the MSFT study. After completing
MSFT fundamentals or the screener here, notebook 03 can publish the corresponding
output-only variant from its exact saved execution.

In [ ]:
# Close this Python client and its connections.
# The services and the saved studies, executions and answers remain available.
await epoch.aclose()